# TFXcan on Google Colab

This notebook takes as input GWAS summary statistics, and some other pre-formatted data that will be downloaded and returns TWAS-like TF binding–trait associations. These are the same steps and
scripts as the [TFXcan-snakemake](https://github.com/hakyimlab/TFXcan-snakemake) pipeline, run one
at a time.

| Step | What happens | Script |
|---|---|---|
| 1 | Settings, Google Drive, software, reference data | |
| 2 | Load your GWAS and standardize the columns | (this notebook) |
| 3 | Check the genome build is hg38; lift over from hg19 if not | (this notebook) |
| 4 | Match variants to the 1000 Genomes EUR reference; split by chromosome | `process_summary_statistics.R` |
| 5 | Pick the top SNP in each LD block (the loci) | `select_top_snps.R`, `collect_topsnps_results.R` |
| 6 | Enformer: predict 5,313 epigenomic features per locus for each individual's two haplotypes | `workflow/enformer/enformer_predict.py` |
| 7 | Enpact: turn the features into TF binding scores | `enpact_predict.py` |
| 8 | Train SNP-based models of TF binding (lEnpact) | PredictDb-nextflow |
| 9 | Association test: S-PrediXcan with the GWAS | `SPrediXcan.py` |
| 10 | Collect and view the results | `collect_summaryTFXcan_results.R` |

**Before you start**

- Choose *Runtime → Change runtime type → GPU*. An A100 or L4 is best but not free; a free T4 works but is slow.
- Rough Enformer time for 100 loci × 100 individuals: about 1.5 h on an A100, 3.5–4.5 h on an L4, 8–14 h on a T4.
- Training (step 8) takes minutes to an hour **per TF model**. For about 734 TF-tissue pairs, you might need to calculate what you need. 
- Outputs are written to your Google Drive. If Colab disconnects, re-run the notebook from the top:
  finished steps are skipped.

> **Status: success.** Runs successfully for 5 GWAS loci for 3 TF-tissue pairs, linearized on 100 individuals. 

## 1. Setup

### 1a. Your settings

This is the only cell you should need to edit. It is set up for a small demo GWAS (GWAS Catalog
GCST90428427), read straight from Box. You can include a direct link to your own GWAS data.
At p ≤ 1e-6 it has about 5 loci for a quick, minimal test.

For your own GWAS, change `PHENOTYPE`, `GWAS_FILE` (a Drive path or a URL) and probably
`GWAS_PVALUE_THRESHOLD` back to 5e-8.

In [ ]:
from datetime import date

# ---- your GWAS ----
PHENOTYPE = "minimal"    # short name: letters, numbers, underscores
# demo GWAS; for yours, e.g. "/content/drive/MyDrive/tfxcan/gwas/my_trait.sumstats.tsv.gz"
GWAS_FILE = "https://uchicago.box.com/shared/static/pbcazpnbzv2q2qe27c1g3ci030jiquin.gz"
GWAS_SEP = "\t"          # "\t" for tab-separated, "," for csv, r"\s+" for space-separated

# TFXcan name -> the column name in YOUR file
COLUMNS = {
    "chrom": "chromosome",
    "pos":   "base_pair_location",
    "ref":   "other_allele",     # the non-effect allele
    "alt":   "effect_allele",    # the allele beta (and the z-score) refer to
    "pval":  "p_value",
    "beta":  "beta",
    "se":    "standard_error",
    "rsid":  "rsid",             # optional: set to None if your file has no rsIDs
}

# ---- choosing loci ----
GWAS_PVALUE_THRESHOLD = 1e-6   # demo: ~5 loci (1e-5: ~19); 5e-8 is the usual choice for your own GWAS
SELECTION_METHOD = "linkage"   # linkage = top SNP per LD block; topSNPs = top SELECT_N_SNPS per chromosome
SELECT_N_SNPS = 100            # only used by topSNPs
LIMIT_NUMBER_OF_LOCI = 100     # at most this many loci genome-wide; None = all
RANK_BY = "zscore"             # zscore (largest |z|) or pval

# ---- Enformer ----
N_INDIVIDUALS = -1             # 1000 Genomes EUR individuals to predict for; -1 = all 100 in the reference data

# ---- TF binding models ----
# Enpact models to test, e.g. ["AR_Prostate", "CTCF_Blood"]; "all" = all 734 (too slow for Colab)
ENPACT_MODELS = ["AR_Prostate", "CTCF_Blood", "CTCF_Liver"]

RUN_NAME = PHENOTYPE
RUN_DATE = date.today().isoformat()   # set a fixed date, e.g. "2026-10-07", to resume an earlier run

### 1b. Paths and Google Drive

- **Run outputs** go to Drive, so they survive disconnects.
- **Reference data and software** go on Colab's local disk. It is faster, but it is wiped when the
  session ends, so they are fetched again each session.

In [ ]:
import os, sys, glob, subprocess, shutil
from google.colab import drive
drive.mount("/content/drive")

RUNMETA = f"{RUN_NAME}_{RUN_DATE}"

# outputs (Drive)
DRIVE_DIR = "/content/drive/MyDrive/tfxcan"
RUN_DIR = f"{DRIVE_DIR}/runs/{RUNMETA}"
DIRS = {
    "sumstats":    f"{RUN_DIR}/sumstats",                 # your harmonized GWAS
    "processed":   f"{RUN_DIR}/processed_sumstats/{PHENOTYPE}",
    "filtering":   f"{RUN_DIR}/filtering/{PHENOTYPE}",
    "collection":  f"{RUN_DIR}/collection",
    "diagnostics": f"{RUN_DIR}/diagnostics",
    "enformer":    f"{RUN_DIR}/enformer",                 # config + per-locus results
    "aggregated":  f"{RUN_DIR}/aggregated_predictions",
    "predictdb":   f"{RUN_DIR}/predictdb.temp/{PHENOTYPE}",
    "lenpact":     f"{RUN_DIR}/lEnpact/{PHENOTYPE}",
    "summary":     f"{RUN_DIR}/summaryTFXcan/{PHENOTYPE}",
    "output":      f"{RUN_DIR}/output",
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

# software and reference data (local disk)
SOFTWARE_DIR = "/content/software"
REF_DIR = "/content/tfxcan_ref"
NF_WORK_DIR = "/content/nf_work"   # Nextflow scratch; local because it is many small files
TFXCAN = f"{SOFTWARE_DIR}/TFXcan-snakemake"
METAXCAN = f"{SOFTWARE_DIR}/MetaXcan"
R_ENV = "/content/envs/tfxcan-r"        # R, Nextflow, Java
IMLAB_ENV = "/content/envs/imlabtools"  # Python 3.7 for MetaXcan

REF = {
    "fasta":          f"{REF_DIR}/hg38_fasta/Homo_sapiens_assembly38.fasta",
    "enformer_model": f"{REF_DIR}/enformer_weights",
    "vcf_pattern":    f"{REF_DIR}/1000G/vcf/ALL.{{chrom}}.shapeit2_integrated_SNPs_v2a_27022019.GRCh38.phased.vcf.gz",
    "snp_annot":      f"{REF_DIR}/1000G/EUR/EUR.snp_annot.txt",
    "geno":           f"{REF_DIR}/1000G/EUR/EUR.geno.txt",
    # the 100 EUR individuals in the VCFs and genotypes; Enformer and training must use the same ones
    "individuals":    f"{REF_DIR}/1000G/EUR/EUR_individuals.colab100.txt",
    "chain_hg19_hg38": f"{REF_DIR}/liftover/hg19ToHg38.over.chain.gz",
}
# small files that ship with the TFXcan repository
REPO = {
    "ld_blocks":   f"{TFXCAN}/metadata/EUR_LDBlocks.hg38.updated.bed",
    "weights":     f"{TFXCAN}/weights/ENPACT_734_2025-04-24.compiled_weights.with_intercept.lambda.1se.txt.gz",
}
print("run folder:", RUN_DIR)

### 1c. Install software

This takes about 10–15 minutes. Two separate environments are needed:

- **R + Nextflow + Java** for the processing scripts and model training. Nextflow is pinned to
  24.04.4: from 26.04 on it rejects the PredictDb-nextflow config ("Config parsing failed").
- **Python 3.7** for MetaXcan, which needs the older pandas/numpy it was written against.

Enformer runs with the TensorFlow that comes with Colab.

In [ ]:
%%bash
set -euo pipefail
cd /content
apt-get -qq update > /dev/null
apt-get -qq install -y pigz parallel tabix > /dev/null
pip -q install pyfaidx cyvcf2 pyliftover tables pyyaml

# micromamba for the two conda environments
if [ ! -x /usr/local/bin/micromamba ]; then
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /usr/local bin/micromamba
fi
if [ ! -d /content/envs/tfxcan-r ]; then
  micromamba create -y -q -p /content/envs/tfxcan-r -c conda-forge -c bioconda \
    r-base=4.3 r-tidyverse r-data.table r-r.utils r-optparse r-glue r-bigsnpr \
    r-glmnet r-rsqlite r-reshape2 nextflow=24.04.4 openjdk=17 pigz
fi
if [ ! -d /content/envs/imlabtools ]; then
  micromamba create -y -q -p /content/envs/imlabtools -c conda-forge \
    python=3.7 numpy=1.18 scipy=1.4 pandas=0.25.3 statsmodels=0.11 patsy h5py=2.10
fi

# code
mkdir -p /content/software && cd /content/software
[ -d TFXcan-snakemake ] || git clone -q -b main https://github.com/hakyimlab/TFXcan-snakemake.git
[ -d MetaXcan ] || git clone -q https://github.com/hakyimlab/MetaXcan.git
echo "software ready"

In [ ]:
import pandas as pd, numpy as np

CPUS = os.cpu_count()
import psutil
MEM_GB = int(psutil.virtual_memory().total / 1e9 * 0.8)   # leave some room for the notebook

# environment for R / Nextflow commands: their bin folder first on PATH
R_ENVIRON = {k: v for k, v in os.environ.items() if k != "R_HOME"}   # use the env's R, not Colab's
R_ENVIRON.update(PATH=f"{R_ENV}/bin:{os.environ['PATH']}", JAVA_HOME=R_ENV, NXF_DISABLE_CHECK_LATEST="true")
RSCRIPT = f"{R_ENV}/bin/Rscript"

def run(cmd, env=None, cwd=None):
    # run a shell command, show its output as it goes, and stop the notebook if it fails
    print(f"$ {cmd}\n", flush=True)
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash", env=env, cwd=cwd,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"command failed with exit code {p.returncode}")

def gpu_name():
    try:
        return subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                              capture_output=True, text=True).stdout.strip() or None
    except FileNotFoundError:
        return None

print(f"{CPUS} CPUs, {MEM_GB} GB usable memory, GPU: {gpu_name()}")

### 1d. Reference data

The reference data come as three archives on Box. Each is downloaded and unpacked into `REF_DIR`
the first time a step needs it. The 1000 Genomes data are cut down to 100 EUR individuals
(see `misc.qmd`).

| Archive | Contents | Size | Needed in |
|---|---|---|---|
| `genome.tar.gz` | hg38 FASTA + `.fai` | ~1 GB download, 3.2 GB unpacked | steps 3, 6 |
| `enformer.tar` | Enformer SavedModel | 1 GB | step 6 |
| `1000G_vcfs_geno.tar` | phased VCFs (+ `.tbi`), text genotypes, SNP annotation, the 100 individuals | 2.7 GB | steps 4, 6, 8 |
| hg19→hg38 chain (UCSC) | | 0.2 MB | step 3, only if your GWAS is hg19 |


In [ ]:
BOX = "https://uchicago.box.com/shared/static"
# archive -> (URL, the REF entries it provides); each unpacks into REF_DIR
ARCHIVES = {
    "genome":   (f"{BOX}/5b6wxzpk0hqlisarhwxpiik1ize6qphc.gz",  ["fasta"]),
    "enformer": (f"{BOX}/wle5kv2dyaj61m071j18efazzeb8ke3x.tar", ["enformer_model"]),
    "1000G":    (f"{BOX}/x3phh5e7y9dzmw4j26cxb357nstg2qig.tar", ["vcf_pattern", "snp_annot", "geno", "individuals"]),
}
CHAIN_URL = "https://hgdownload.soe.ucsc.edu/goldenPath/hg19/liftOver/hg19ToHg38.over.chain.gz"

def fetch(name):
    # make sure a reference file is on local disk, downloading its archive if needed; returns REF[name]
    if name == "chain_hg19_hg38":
        if not os.path.exists(REF[name]):
            os.makedirs(os.path.dirname(REF[name]), exist_ok=True)
            run(f"wget -q -O '{REF[name]}.part' '{CHAIN_URL}' && mv '{REF[name]}.part' '{REF[name]}'")
        return REF[name]
    archive = next(a for a, (_, names) in ARCHIVES.items() if name in names)
    url = ARCHIVES[archive][0]
    done = f"{REF_DIR}/.{archive}.done"   # written only after a complete unpack
    if not os.path.exists(done):
        os.makedirs(REF_DIR, exist_ok=True)
        z = "z" if url.endswith(".gz") else ""
        # stream straight into tar: no second copy on disk. The archives were made on a Mac, so skip
        # its ._* metadata files and the extra header fields GNU tar does not know.
        run(f"set -o pipefail; wget -q -O - '{url}' | tar -x{z}f - -C '{REF_DIR}' "
            f"--exclude='._*' --warning=no-unknown-keyword && touch '{done}'")
    return REF[name]


## 2. Load your GWAS

TFXcan needs, for each variant: chromosome, position, the two alleles, the p-value, beta and its
standard error. Only autosomal single-nucleotide variants are kept: chromosomes 1–22, single A/C/G/T alleles.
The z-score is computed as beta / se.

In [ ]:
use = {v: k for k, v in COLUMNS.items() if v is not None}
gwas = pd.read_csv(GWAS_FILE, sep=GWAS_SEP, usecols=list(use), dtype=str).rename(columns=use)
n_raw = len(gwas)

gwas["chrom"] = gwas["chrom"].str.replace("^chr", "", regex=True, case=False)
gwas = gwas[gwas["chrom"].isin([str(i) for i in range(1, 23)])]
for col in ["pos", "pval", "beta", "se"]:
    gwas[col] = pd.to_numeric(gwas[col], errors="coerce")
gwas["ref"] = gwas["ref"].str.upper()
gwas["alt"] = gwas["alt"].str.upper()
acgt = gwas["ref"].isin(list("ACGT")) & gwas["alt"].isin(list("ACGT")) & (gwas["ref"] != gwas["alt"])
gwas = gwas[acgt].dropna(subset=["pos", "pval", "beta", "se"])
gwas = gwas[gwas["se"] > 0]
gwas["chrom"] = gwas["chrom"].astype(int)
gwas["pos"] = gwas["pos"].astype(int)
gwas["zscore"] = gwas["beta"] / gwas["se"]

print(f"{n_raw:,} rows read; {len(gwas):,} autosomal SNVs kept")
print(f"{(gwas['pval'] <= GWAS_PVALUE_THRESHOLD).sum():,} SNVs pass p <= {GWAS_PVALUE_THRESHOLD}")
gwas.head()

## 3. Check the genome build

For a random sample of variants, look up the base at that position in the hg38 genome. Then check
whether it equals one of the variant's two alleles.

- **hg38:** nearly all variants match. Expect over 95%; the rest are mostly errors in the file.
- **Another build:** the position points to an unrelated base, so only about half match by chance.

If the hg38 match is low, the same sample is lifted from hg19 to hg38 and checked again. If that
matches, the whole file is lifted over. Variants that don't map, or that land on another chromosome
or the opposite strand, are dropped.

In [ ]:
import pyfaidx

fasta = pyfaidx.Fasta(fetch("fasta"))

def allele_match_rate(df, n=20000, seed=1):
    # fraction of sampled variants whose hg38 base equals ref or alt; also how often it equals ref
    s = df.sample(min(n, len(df)), random_state=seed)
    hits_any = hits_ref = total = 0
    for c, p, r, a in zip(s["chrom"], s["pos"], s["ref"], s["alt"]):
        contig = fasta[f"chr{c}"]
        if not 1 <= p <= len(contig):
            continue
        base = contig[p - 1].seq.upper()
        total += 1
        hits_any += base in (r, a)
        hits_ref += base == r
    return hits_any / total, hits_ref / total

match_hg38, ref_is_ref = allele_match_rate(gwas)
print(f"hg38: {match_hg38:.1%} of sampled variants match one of their alleles")
print(f"      {ref_is_ref:.1%} have the hg38 base as the 'ref' allele")

In [ ]:
from pyliftover import LiftOver

def liftover_hg19_to_hg38(df):
    lo = LiftOver(fetch("chain_hg19_hg38"))
    new_pos = []
    for c, p in zip(df["chrom"], df["pos"]):
        hits = lo.convert_coordinate(f"chr{c}", int(p) - 1)   # pyliftover is 0-based
        ok = hits and hits[0][0] == f"chr{c}" and hits[0][2] == "+"
        new_pos.append(hits[0][1] + 1 if ok else -1)
    out = df.assign(pos=new_pos)
    return out[out["pos"] > 0]

BUILD_OK = 0.95
if match_hg38 >= BUILD_OK:
    print("Your GWAS looks like hg38. Nothing to do.")
else:
    lifted_sample = liftover_hg19_to_hg38(gwas.sample(min(20000, len(gwas)), random_state=1))
    match_lifted, _ = allele_match_rate(lifted_sample)
    print(f"after lifting a sample from hg19: {match_lifted:.1%} match")
    if match_lifted < BUILD_OK:
        raise ValueError("Your GWAS matches neither hg38 nor hg19. Check the build, and that "
                         "COLUMNS points to the right chromosome, position and allele columns.")
    print(f"Your GWAS looks like hg19; lifting all {len(gwas):,} variants to hg38 (a few minutes)...")
    n_before = len(gwas)
    gwas = liftover_hg19_to_hg38(gwas)
    print(f"{len(gwas):,} variants lifted; {n_before - len(gwas):,} dropped")
    match_hg38, _ = allele_match_rate(gwas)
    print(f"hg38 match after liftover: {match_hg38:.1%}")

Next, write the harmonized GWAS in the format the TFXcan scripts read:
`chrom pos variant_id ref alt pval zscore beta se [rsid]`.

In [ ]:
gwas["variant_id"] = (gwas["chrom"].astype(str) + "_" + gwas["pos"].astype(str) + "_"
                      + gwas["ref"] + "_" + gwas["alt"])
gwas = gwas.drop_duplicates("variant_id")
cols = ["chrom", "pos", "variant_id", "ref", "alt", "pval", "zscore", "beta", "se"]
cols += ["rsid"] if "rsid" in gwas.columns else []
HARMONIZED_GWAS = f"{DIRS['sumstats']}/{PHENOTYPE}.gwas_sumstats.hg38.processed.txt.gz"
gwas[cols].sort_values(["chrom", "pos"]).to_csv(HARMONIZED_GWAS, sep="\t", index=False)
print("wrote", HARMONIZED_GWAS)

## 4. Match to the 1000 Genomes EUR reference

Variants are matched to the EUR reference panel by position and alleles (`bigsnpr::snp_match`),
with flipped and reversed alleles allowed. Effects are aligned to the reference allele and the
output is split by chromosome. The trained models and their LD come from these individuals, so
this step decides which GWAS variants can be used.

In [ ]:
if not glob.glob(f"{DIRS['processed']}/chr*.sumstats.txt.gz"):
    run(f"{RSCRIPT} {TFXCAN}/workflow/process/process_summary_statistics.R "
        f"--summary_stats_file {HARMONIZED_GWAS} --output_folder {DIRS['processed']} "
        f"--annotation_file {fetch('snp_annot')} "
        f"--diagnostics_file {DIRS['diagnostics']}/{PHENOTYPE}.gwas_diagnostics.summary "
        f"--pvalue_threshold {GWAS_PVALUE_THRESHOLD}", env=R_ENVIRON)

CHROMS = sorted(int(os.path.basename(f).split(".")[0][3:])
                for f in glob.glob(f"{DIRS['processed']}/chr*.sumstats.txt.gz"))
n_matched = sum(len(pd.read_table(f"{DIRS['processed']}/chr{c}.sumstats.txt.gz", usecols=[0])) for c in CHROMS)
print(f"{n_matched:,} of {len(gwas):,} variants matched the reference, on {len(CHROMS)} chromosomes")

## 5. Choose the loci

In each LD block that has a variant at or below the p-value threshold, the top variant is kept
(`linkage`). Ranking is by |z| or p-value, as set in `RANK_BY`. Then at most `LIMIT_NUMBER_OF_LOCI`
loci are kept genome-wide. Each locus is the region Enformer will predict around.

In [ ]:
LOCI_FILE = f"{DIRS['collection']}/{PHENOTYPE}.EnformerLoci.topSNPs.txt"
FILTERED_GWAS = f"{DIRS['collection']}/{PHENOTYPE}.filteredGWAS.topSNPs.txt.gz"

if not os.path.exists(LOCI_FILE):
    # one R process per chromosome, as in the pipeline
    sumstats = f"{DIRS['processed']}/chr{{}}.sumstats.txt.gz"
    diag = f"{DIRS['diagnostics']}/{PHENOTYPE}.chr{{}}.topSNPs_diagnostics.summary"
    run(f"printf '%s\\n' {' '.join(map(str, CHROMS))} | parallel -j {CPUS} "
        f"\"{RSCRIPT} {TFXCAN}/workflow/process/select_top_snps.R --chromosome {{}} --sumstats {sumstats} "
        f"--LDBlocks_info {REPO['ld_blocks']} --output_folder {DIRS['filtering']} --phenotype {PHENOTYPE} "
        f"--diagnostics_file {diag} --selection_method {SELECTION_METHOD} --select_n_snps {SELECT_N_SNPS} "
        f"--rank_by {RANK_BY}\"", env=R_ENVIRON)
    limit = f"--limit_number_of_loci {LIMIT_NUMBER_OF_LOCI}" if LIMIT_NUMBER_OF_LOCI else ""
    run(f"{RSCRIPT} {TFXCAN}/workflow/process/collect_topsnps_results.R --selection_dir {DIRS['filtering']} "
        f"--phenotype {PHENOTYPE} --filtered_sumstats {FILTERED_GWAS} --enformer_loci {LOCI_FILE} "
        f"--rank_by {RANK_BY} {limit}", env=R_ENVIRON)

loci = pd.read_table(LOCI_FILE, header=None, sep=r"\s+")[0].drop_duplicates().tolist()
if not loci:
    raise ValueError("No loci passed the threshold. Try a larger GWAS_PVALUE_THRESHOLD.")
print(f"{len(loci)} loci, e.g. {loci[:3]}")

## 6. Enformer predictions

For each locus and each individual, both haplotype sequences are built from the phased 1000 Genomes
VCFs and run through Enformer. The bins around the locus are averaged, and the two haplotypes are
summed, giving 5,313 features per locus per individual.

The pipeline runs one cluster job per locus. Here, one process predicts every locus, so the model
loads only once. Each finished locus is saved to Drive, so a re-run after a disconnect skips it.

In [ ]:
import yaml

# rough time: seconds per sequence measured on an A40 (0.51 s), scaled for other GPUs
SEC_PER_SEQ = {"A100": 0.29, "L4": 0.7, "A40": 0.51, "T4": 2.0}
gpu = gpu_name() or "none"
sps = next((v for k, v in SEC_PER_SEQ.items() if k in gpu), None)
n_all = sum(1 for line in open(fetch("individuals")) if line.strip())
n_ind = n_all if N_INDIVIDUALS == -1 else min(N_INDIVIDUALS, n_all)
n_seq = len(loci) * n_ind * 2
if sps:
    print(f"{len(loci)} loci x {n_ind} individuals x 2 haplotypes = {n_seq:,} sequences; "
          f"about {n_seq * sps / 3600:.1f} h on this {gpu}")
else:
    print(f"GPU: {gpu}. Without a GPU Enformer is far too slow; change the runtime type.")

fetch("vcf_pattern")

ENFORMER_BASENAME = f"{PHENOTYPE}.{RUNMETA}.processed"
ENFORMER_CONFIG = f"{DIRS['enformer']}/enformer_config_{RUN_NAME}_{PHENOTYPE}.yaml"
with open(ENFORMER_CONFIG, "w") as f:   # same settings as the pipeline's write_enformer_config rule
    yaml.dump({
        "loci_file": LOCI_FILE,
        "individuals": REF["individuals"],
        "n_individuals": N_INDIVIDUALS,
        "vcf_pattern": REF["vcf_pattern"],
        "fasta_file": REF["fasta"],
        "model_path": fetch("enformer_model"),
        "pad_bins": 1,
        "aggregation": "mean",
        "batch_size": 1,
        "devices": "auto",
        "output_dir": DIRS["aggregated"],
        "output_basename": ENFORMER_BASENAME,
        "loci_dir": f"{DIRS['enformer']}/loci",
    }, f, sort_keys=False)

ENFORMER_MATRIX = f"{DIRS['aggregated']}/{ENFORMER_BASENAME}.matrix.h5.gz"
ENFORMER_METADATA = f"{DIRS['aggregated']}/{ENFORMER_BASENAME}.metadata.tsv"
if not os.path.exists(ENFORMER_MATRIX):
    run(f"python3 {TFXCAN}/workflow/enformer/enformer_predict.py --config {ENFORMER_CONFIG}")   # predict all loci, then merge
print("Enformer features:", ENFORMER_MATRIX)

## 7. Enpact: TF binding scores

Each Enpact model is a linear combination of the 5,313 Enformer features that predicts binding of
one TF in one tissue. This step scores every locus × individual with the models chosen in
`ENPACT_MODELS`. It writes one score file and one annotation file per model; these are the
inputs for training in step 8.

In [ ]:
weights = pd.read_table(REPO["weights"])   # column 1 = feature; row 1 = intercept
available = weights.columns[1:].tolist()
MODELS = available if ENPACT_MODELS == "all" else list(ENPACT_MODELS)
unknown = sorted(set(MODELS) - set(available))
if unknown:
    raise ValueError(f"not in the Enpact weights: {unknown}. Available: {len(available)} models, e.g. {available[:5]}")

SELECTED_WEIGHTS = f"{RUN_DIR}/enpact_weights.selected.txt.gz"
weights[["feature"] + MODELS].to_csv(SELECTED_WEIGHTS, sep="\t", index=False)

ENPACT_BASENAME = f"{DIRS['predictdb']}/{PHENOTYPE}"
if not all(os.path.exists(f"{ENPACT_BASENAME}.{m}.enpact_scores.txt") for m in MODELS):
    run(f"python3 {TFXCAN}/workflow/process/enpact_predict.py --matrix {ENFORMER_MATRIX} "
        f"--weights {SELECTED_WEIGHTS} --metadata {ENFORMER_METADATA} --split "
        f"--output_basename {ENPACT_BASENAME} --subset_of_loci {LOCI_FILE}")
print(f"Enpact scores for {len(MODELS)} models in {DIRS['predictdb']}")

## 8. Train SNP-based TF binding models (lEnpact)

For each TF model, an elastic net is fit with nested cross-validation (PredictDb-nextflow). It
predicts the Enpact score at each locus from the genotypes of nearby SNPs in the same 1000 Genomes
individuals. The result is a PrediXcan-style database of SNP weights plus a SNP covariance file,
which is what S-PrediXcan needs.

This stage uses nested crossvalidation. You can remove that option and it should make training go faster.

This is the slow part on Colab. The pipeline runs one 36-hour cluster job per model; here the
models are trained one after another with the CPUs Colab gives you.

In [ ]:
NEXTFLOW_MAIN = f"{TFXCAN}/workflow/PredictDb-nextflow/main.nf"
geno, snp_annot = fetch("geno"), fetch("snp_annot")

def lenpact_paths(model):
    out = f"{DIRS['lenpact']}/{model}"
    db = f"{out}/models/filtered_db/predict_db_{PHENOTYPE}_filtered"
    return out, f"{db}.db", f"{db}.txt.gz", f"{out}/models/filtered_db/Covariances.varID.txt.gz"

for i, model in enumerate(MODELS, 1):
    out, model_db, cov_raw, cov = lenpact_paths(model)
    if os.path.exists(model_db) and os.path.exists(cov):
        print(f"[{i}/{len(MODELS)}] {model}: already trained")
        continue
    print(f"[{i}/{len(MODELS)}] training {model}")
    os.makedirs(out, exist_ok=True)
    nf_dir = f"{NF_WORK_DIR}/{model}"   # Nextflow's .nextflow/ cache and log; a database, so not on Drive
    os.makedirs(nf_dir, exist_ok=True)
    # same arguments as workflow/predictdb/generate_snp_predictors.sbatch, but run locally (no -profile midway3)
    run(f"nextflow run {NEXTFLOW_MAIN} "
        f"--gene_annotation {ENPACT_BASENAME}.{model}.annotation.txt --snp_annotation {snp_annot} "
        f"--genotype {geno} --gene_exp {ENPACT_BASENAME}.{model}.enpact_scores.txt "
        f"--prefix {PHENOTYPE} --outdir {out}/models/ --nested_cv "
        f"--max_cpus {CPUS} --max_memory '{MEM_GB}.GB' "
        f"-w {NF_WORK_DIR}/{model}/work -resume", env=R_ENVIRON, cwd=nf_dir)
    # rename the SNP ids in the covariance file (1:100:A:G -> 1_100_A_G) to match the GWAS
    run(f"bash {TFXCAN}/workflow/src/format_covariances.sbatch {cov_raw} {cov}", env=R_ENVIRON)
    os.remove(cov_raw)   # the pipeline also deletes it (temp) once the renamed copy is checked

## 9. Association test (S-PrediXcan)

For each TF model, S-PrediXcan combines the SNP weights with the GWAS z-scores and the SNP
covariance. The result is a z-score and p-value for the association between predicted TF binding
at each locus and the trait.

In [ ]:
for i, model in enumerate(MODELS, 1):
    _, model_db, _, cov = lenpact_paths(model)
    result = f"{DIRS['summary']}/{model}-{PHENOTYPE}.enpactScores.spredixcan.csv"
    if os.path.exists(result):
        continue
    print(f"[{i}/{len(MODELS)}] {model}")
    # same arguments as workflow/process/summary_TFXcan.sbatch
    run(f"{IMLAB_ENV}/bin/python {METAXCAN}/software/SPrediXcan.py "
        f"--model_db_path {model_db} --model_db_snp_key varID --covariance {cov} "
        f"--gwas_folder {DIRS['processed']} --gwas_file_pattern '.*.sumstats.txt.gz' "
        f"--chromosome_column chrom --snp_column varID --effect_allele_column alt "
        f"--non_effect_allele_column ref --zscore_column zscore --keep_non_rsid "
        f"--output_file {result} --verbosity 9 --throw")
print("S-PrediXcan results in", DIRS["summary"])

## 10. Results

All models are combined into one table. Each row is one TF binding site, labeled
`<TF>_<tissue>_<locus>`, with its association z-score and p-value.

In [ ]:
paths_file = f"{DIRS['collection']}/{PHENOTYPE}.summaryTFXcan.paths.txt"
with open(paths_file, "w") as f:
    f.writelines(f"{DIRS['summary']}/{m}-{PHENOTYPE}.enpactScores.spredixcan.csv\n" for m in MODELS)

TFXCAN_RESULTS = f"{DIRS['output']}/{PHENOTYPE}.enpactScores.{RUN_DATE}.spredixcan.txt"
run(f"{RSCRIPT} {TFXCAN}/workflow/process/collect_summaryTFXcan_results.R "
    f"--input_files_pattern {paths_file} --phenotype {PHENOTYPE} --output_file {TFXCAN_RESULTS}", env=R_ENVIRON)

res = pd.read_table(TFXCAN_RESULTS)
res["bonferroni_significant"] = res["pvalue"] <= 0.05 / len(res)
print(f"{len(res):,} TF binding sites tested; {res['bonferroni_significant'].sum()} pass Bonferroni (0.05 / {len(res):,})")
print("results:", TFXCAN_RESULTS)
res.sort_values("pvalue").head(20)

## Same run with Snakemake

On a cluster, the whole notebook corresponds to a single pipeline call:

```bash
snakemake -s snakefile.smk --configfile config/base_config.yaml config/<your_run>.yaml \
    --profile profiles/simple/ --resources load=100
```

The pipeline also has a factorization step that groups the associated TF binding sites into programs
(`workflow/factorize/`). It is not included here.